# Time-Series Forecasting — Walk-Forward CV, Lag Features, and Baselines

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb16_time_series_forecasting_student.ipynb)


> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. Complete both to receive participation credit.

---


## Learning Objectives

By the end of this notebook, you will be able to:

1. Distinguish a forecasting problem from a generic supervised-learning problem and choose the right evaluation protocol.
2. Run a structured time-series EDA — time plot, seasonal sub-series, decomposition, autocorrelation — on a real labor-market dataset.
3. Build a **time-respecting** 60/20/20 train/validation/test split where the test window is the most recent slice of history.
4. Run **walk-forward cross-validation** with `TimeSeriesSplit` instead of k-fold CV (which would shuffle time and leak the future into the past).
5. Compare four classical forecasting benchmarks (Mean, Naive, Seasonal-Naive, Drift) against a learned **lag-feature linear regression** on identical CV folds.
6. Add **regularization** (Ridge) to the lag-feature linear model and decide whether it earns its place via the Student's *t* 95% CI overlap rule.
7. Open the locked test window in a **one-shot evaluation ceremony**, mirroring nb14's protocol but adapted to time.


## Setup

Import the libraries we will use across the notebook. Two new tools today: `TimeSeriesSplit` (walk-forward CV) and `STL` (seasonal-trend decomposition from `statsmodels`). Everything else is the Week-1 analytics toolkit you have used since nb01.


In [ ]:
# Setup Cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from scipy.stats import t as student_t

warnings.filterwarnings("ignore")

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.precision', 3)
sns.set_style("whitegrid")

print("Setup complete!")
print(f"RANDOM_SEED = {RANDOM_SEED}")


**Reading the output:** A clean `Setup complete!` confirms the libraries we need are available. If `statsmodels` or `sklearn` are missing, restart the runtime — they ship with Colab and rarely need explicit installation.

---


## 1. Why This Matters: Forecasting US Retail Employment

The **US Bureau of Labor Statistics** publishes monthly employment counts for every major industry. A **state-level workforce planner** uses those numbers to forecast next year's retail labor demand:

> *"I need a defensible forecast of US retail-sector employment one year out — with a confidence interval the legislature can read. Last year's headline number is not enough; I need the model and the diagnostics."*

This is **not** the kind of problem we solved in nb01–nb15. There, every row was an independent observation and a 60/20/20 random split was the right protocol. Here, the rows are months in a sequence — the **order matters**, and shuffling them would let the model peek at the future during training (a classic data leak). The fix is structural: the test window is always the **most recent** slice of history, and cross-validation walks forward in time.

This notebook ports the **Week-1 analytics workflow** (EDA → split → baselines → linear features → regularization) to the time-series setting, threading the same structural rule throughout: *the future cannot leak into the past.*

**A question that often comes up here:** *"Why isn't this just nb14 with a different metric?"* Two reasons. First, k-fold CV with shuffled rows would let row 50 be in the training fold and row 49 in the validation fold — the model would see a future month while learning to predict an earlier one. That defeats the entire idea of forecasting. Second, employment series have **seasonality** (holiday hiring) and **long-run trend** (decades of structural growth), so a feature engineered as "value 12 months ago" is structurally meaningful in a way that "row 12 in the dataset" is not.


## 2. Load the Data and Sanity Checks

We use the **US Employment dataset** from the *Forecasting: Principles and Practice* (FPP) textbook. It contains monthly employment counts (in thousands) for every BLS major industry from 1939 onward. We filter to the **"Retail Trade"** series — the workforce planner's target — and parse the date column.


In [ ]:
# Load the full dataset (≈8.6 MB) directly from the course's GitHub raw URL
DATA_URL = (
    "https://raw.githubusercontent.com/davi-moreira/"
    "2026Summer_predictive_analytics_purdue_MGMT474/main/"
    "lecture_slides/08_time_series/data/us_employment.csv"
)
us_employment = pd.read_csv(DATA_URL, parse_dates=["ds"])

# Filter to the Retail Trade series and drop other columns
df = (
    us_employment.query('unique_id == "Retail Trade"')
    .loc[:, ["ds", "y"]]
    .sort_values("ds")
    .reset_index(drop=True)
)

print(f"Rows: {len(df):,}")
print(f"Date range: {df['ds'].min().date()}  ->  {df['ds'].max().date()}")
print(f"Missing values: {df.isna().sum().to_dict()}")
print()
print(df.head())


**Reading the output:**

You should see roughly **960 rows** spanning **1939-01 through 2019-09** (80 years of monthly data) with **zero missing values**. The two columns are `ds` (date stamp, parsed as `datetime64`) and `y` (employment in thousands). This is the cleanest possible time-series setup: a single numeric target indexed by a regular monthly date.

**A question that often comes up here:** *"Why is `unique_id` a string column?"* The original dataset is in long format — one row per (industry × month). The `unique_id` column tags each row with its industry. The lecture's `_08_time_series.ipynb` walks through several industries; we filter to a single one so the analysis stays focused.


In [ ]:
# Distribution of y (one-number sanity check)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["y"], bins=40, color="#1f77b4", edgecolor="white")
ax.set_xlabel("US Retail Trade employment (thousands)")
ax.set_ylabel("Count of monthly observations")
ax.set_title("Distribution of monthly US retail employment, 1939–2019")
plt.tight_layout()
plt.show()

print(df["y"].describe().round(0))


**Reading the output:**

The histogram is wide and bimodal-looking because the series spans 80 years of structural growth — the early decades sit in the 5,000–8,000 range while the later decades sit in the 14,000–16,000 range. The five-number summary confirms a **2.7×** range from minimum to maximum, which is a strong hint that the series has **trend**. No outliers, no negatives, no impossible values. Section 3 puts six diagnostic plots behind these numbers — each one answers a specific structural question about the series.

---

## 3. Time-Series EDA — Six Plots, One Story

A time series asks for visual EDA before any modeling. The canonical sequence is: **time plot** (the whole series), **time plot zoomed** (a recent slice), **seasonal sub-series** (per-month box plot), **STL decomposition** (trend + seasonal + remainder), **ACF** (autocorrelation function), **lag-1 scatter** (do consecutive months track each other?). Six plots, one combined story.


### 3.1 Time plot — the whole series

The first plot every forecaster makes. The shape of the curve answers the questions *"is there a trend?"* and *"is there seasonality?"* faster than any statistical test.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["ds"], df["y"], color="#1f77b4", linewidth=0.8)
ax.set_xlabel("Year")
ax.set_ylabel("Employment (thousands)")
ax.set_title("US Retail Trade Employment, 1939–2019 (monthly)")
plt.tight_layout()
plt.show()


**Reading the output:**

Three structural features jump out immediately for the workforce planner. Employment roughly **triples** from 1939 to 2019 — the mid-century post-war growth, the 1990s e-commerce boom, the 2008 dip, and the steady recovery are all visible, confirming a strong **long-run trend**. Layered on top of that trend, small annual ripples mark holiday hiring in November/December and January layoffs — the **seasonal** component. The sharpest disruptions — the 2008–2010 dip, plus smaller dips in 1974, 1980, 1990, and 2001 — trace **recessions** that no purely time-based feature can anticipate. A single forecasting model has to capture all three.

### 3.2 Time plot zoomed — last 10 years

The full series compresses the seasonal pattern into illegibility. Zooming in to the last decade brings it out.


In [ ]:
recent = df[df["ds"] >= "2010-01-01"]
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(recent["ds"], recent["y"], "o-", color="#2ca02c", markersize=4)
ax.set_title("US Retail Trade Employment — 2010–2019 (monthly)")
ax.set_xlabel("Year")
ax.set_ylabel("Employment (thousands)")
plt.tight_layout()
plt.show()


**Reading the output:**

Now the **annual peaks in November/December** (holiday retail hiring) and the **post-holiday troughs in January/February** are unmistakable. Each year is a small wave on top of a slow upward trend. This is exactly the structure that **lag-12 features** will capture in section 8.


### 3.3 Seasonal sub-series box plot

A box plot of `y` grouped by **month-of-year** answers *"how strong and how stable is the seasonal pattern?"* in one figure. If December's box is consistently above June's, the seasonal effect is strong; if every month's box overlaps every other, there is no seasonality to capture.


In [ ]:
df_seasonal = df.copy()
df_seasonal["month"] = df_seasonal["ds"].dt.month
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df_seasonal, x="month", y="y", ax=ax, color="#ff7f0e")
ax.set_title("Seasonal sub-series — employment distribution by month-of-year")
ax.set_xlabel("Month of year")
ax.set_ylabel("Employment (thousands)")
plt.tight_layout()
plt.show()


**Reading the output:**

**December** sits visibly above the rest, then November, then a steady plateau from spring to fall, then the January/February dip. The medians trace a smooth annual cycle, but the **boxes overlap a lot** — that overlap is the long-run trend (80 years of growth) hiding inside the by-month aggregation. The seasonal *shape* is real; the seasonal *amplitude* relative to the trend is moderate.


### 3.4 STL decomposition — trend + seasonal + remainder

**STL (Seasonal-Trend decomposition using Loess)** splits the series into three additive components:

$$y_t = T_t + S_t + R_t$$

— a smooth trend, a periodic seasonal pattern, and what is left over (the "remainder"). It is the time-series analog of "explain the variance and look at the residuals."


In [ ]:
# STL decomposition with monthly period
ts = df.set_index("ds")["y"]
stl = STL(ts, period=12, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
axes[0].plot(ts.index, ts.values, color="black"); axes[0].set_ylabel("y (data)")
axes[1].plot(ts.index, stl.trend, color="#1f77b4"); axes[1].set_ylabel("Trend")
axes[2].plot(ts.index, stl.seasonal, color="#2ca02c"); axes[2].set_ylabel("Seasonal")
axes[3].plot(ts.index, stl.resid, color="#d62728"); axes[3].set_ylabel("Remainder")
axes[3].axhline(0, color="black", linewidth=0.5)
axes[0].set_title("STL Decomposition — US Retail Trade Employment")
plt.tight_layout()
plt.show()


**Reading the output:**

Four panels, four structural facts:

1. **Data** — the raw series.
2. **Trend** — the smooth long-run component. Growth across decades, 2008 dip, gradual recovery.
3. **Seasonal** — the regular annual pattern. Same wave shape repeating every 12 months. The amplitude is small relative to the trend, but it is **consistent**.
4. **Remainder** — what neither trend nor seasonality explains. Spikes around recessions (1974, 2008) — the model cannot anticipate macro shocks from the series alone.

**A question that often comes up here:** *"Should I use additive or multiplicative decomposition?"* Additive when the seasonal amplitude does not grow with the trend; multiplicative when it does. Visually, the seasonal swing here is roughly the same height in 1950 (small absolute employment) as in 2010 (large absolute employment) → additive is the right call. If the seasonal swing was larger in absolute terms when employment was higher, multiplicative would fit better.


### 3.5 Autocorrelation (ACF) plot

The ACF asks *"how strongly does month $t$'s value depend on month $t-k$'s value, for each lag $k$?"*. Tall bars at lags 1, 2, 3 mean **trend / momentum**. A tall bar at lag 12 (and again at 24) means **annual seasonality**.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
plot_acf(ts.values, lags=36, ax=ax, zero=False)
ax.set_title("Autocorrelation function — lags 1 through 36")
ax.set_xlabel("Lag (months)")
plt.tight_layout()
plt.show()


**Reading the output:**

Two patterns jump out. The bars decay slowly across lags 1–24, confirming the strong trend and autocorrelation visible in the time plot — yesterday’s value really is the best naive guess for today. On top of that slow decay, local peaks at lags 12 and 24 flag annual seasonality: December’s value pairs strongly with last December’s, and with two Decembers ago, and so on.

This ACF gives us **direct empirical justification** for the two lag features we will engineer in section 8: `lag1` captures the trend and autocorrelation, `lag12` captures the annual seasonality. The ACF is the diagnostic; the features are the response.

### 3.6 Lag-1 scatter

The simplest forecast in the world is *"next month equals last month."* The lag-1 scatter checks how good that forecast would be: tight cluster along the 45° line means very good; messy cloud means very bad.


In [ ]:
lag1_df = df.assign(lag1=df["y"].shift(1)).dropna()
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(lag1_df["lag1"], lag1_df["y"], s=8, alpha=0.5, color="#9467bd")
lo, hi = lag1_df["y"].min(), lag1_df["y"].max()
ax.plot([lo, hi], [lo, hi], "k--", linewidth=0.8, label="Perfect lag-1 forecast (y = lag1)")
ax.set_xlabel("Employment, t-1 (lag1)")
ax.set_ylabel("Employment, t")
ax.set_title("Lag-1 scatter — does the previous month predict the current month?")
ax.legend()
plt.tight_layout()
plt.show()


**Reading the output:**

The scatter hugs the 45° line tightly — month-to-month change is small relative to the level. This is the visual proof that the **naive forecast** (“next month = last month”) will be a strong baseline. Any model that does not beat it is not earning its keep.

Six plots, one combined story: strong trend, clear annual seasonality, and tight lag-1 autocorrelation. Section 4 translates those structural facts into the single rule that governs every modeling decision below.

---

## 4. The Structural Rule — Never Shuffle, Never Leak the Future

Every static-classification rule we built since nb01 still works in this notebook — **except one**. Rows here are months in a sequence, and shuffling them would let the model peek at the future during training. That single structural change cascades into three downstream changes:

1. **Train/test split**: the test window is the **most recent slice** of history, not a random sample.
2. **Cross-validation**: every fold's training data must come strictly **before** its validation data.
3. **Features**: lag features (last month, 12 months ago) replace random feature engineering.

Sections 5–7 implement each one in order.

---


## 5. Time-Respecting 60/20/20 Split

Week 1 taught the 60/20/20 split. We use the same proportions today, but **order-respecting**: oldest 60% → train, middle 20% → val, most recent 20% → **locked test**. The test window is touched exactly once, in section 10, after the champion is chosen.


In [ ]:
n = len(df)
n_train = int(n * 0.60)
n_val = int(n * 0.20)
n_test = n - n_train - n_val

df_train = df.iloc[:n_train].copy()
df_val   = df.iloc[n_train:n_train + n_val].copy()
df_test  = df.iloc[n_train + n_val:].copy()

print(f"Train: {df_train['ds'].min().date()} -> {df_train['ds'].max().date()}  (n={len(df_train)})")
print(f"Val  : {df_val['ds'].min().date()} -> {df_val['ds'].max().date()}  (n={len(df_val)})")
print(f"Test : {df_test['ds'].min().date()} -> {df_test['ds'].max().date()}  (n={len(df_test)})  [LOCKED]")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_train["ds"], df_train["y"], color="#1f77b4", label="Train")
ax.plot(df_val["ds"],   df_val["y"],   color="#ff7f0e", label="Val")
ax.plot(df_test["ds"],  df_test["y"],  color="#d62728", linestyle="--", label="Test (locked)")
for boundary, color in [(df_train["ds"].max(), "grey"), (df_val["ds"].max(), "grey")]:
    ax.axvline(boundary, color=color, linestyle=":", alpha=0.7)
ax.set_title("Time-Respecting 60/20/20 Split")
ax.legend()
plt.tight_layout()
plt.show()


**Reading the output:**

The plot shows three connected segments. Every model-selection decision below uses train + val (the solid blue + orange portions). The dashed red test window stays locked until the section-10 ceremony.

> **A question that often comes up here:** *“Why 60/20/20 instead of, say, 80/10/10?”* Same reason as nb01: 60% gives the model enough history to fit, 20% gives validation enough power to discriminate between candidates with non-overlapping CIs, and 20% locked test gives the final ceremony enough rows to be a meaningful sample of the recent dynamics. The exact split is conventional; the discipline of holding out a recent slice is structural.

The split defines *which* rows go where; section 6 defines *how* to evaluate models honestly on the training portion using walk-forward cross-validation.

---

## 6. Walk-Forward Cross-Validation with `TimeSeriesSplit`

`TimeSeriesSplit(n_splits=5)` produces 5 folds where every fold's training data comes **before** its validation data, and the training window grows over time. This is the structural fix that makes evaluation honest in a temporal setting.


In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

fig, ax = plt.subplots(figsize=(11, 4))
for fold, (train_idx, val_idx) in enumerate(tscv.split(df_train)):
    ax.plot(train_idx, [fold]*len(train_idx), "s", color="#1f77b4", markersize=3, label="train" if fold == 0 else "")
    ax.plot(val_idx,   [fold]*len(val_idx),   "s", color="#ff7f0e", markersize=3, label="val" if fold == 0 else "")
ax.set_yticks(range(5))
ax.set_yticklabels([f"fold {i+1}" for i in range(5)])
ax.set_xlabel("Month index in training data")
ax.set_title("Walk-Forward CV: train (blue) always precedes val (orange)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


**Reading the output:**

Each row is one CV fold. Blue squares are training months; orange squares are validation months. The critical structural fact: orange always sits to the right of blue — the model never sees a future month while learning to predict an earlier one. The second pattern is that the training window grows across folds — fold 5 trains on roughly 5× the data fold 1 trained on, which mimics what really happens in deployment as history accumulates over time.

With the walk-forward folds in hand, the next question is what to measure on each fold. Section 7 introduces four forecasting metrics and runs the classical benchmarks under all of them.

---

## 7. Forecasting Metrics — Four Benchmarks Under Four Lenses

Before evaluating any model, decide what *good* means. Four metrics show up in every time-series textbook; each answers a slightly different question about forecast quality. Sub-section 7.1 defines each metric with its formula; 7.2 gives a quick "when to use which" guide; the code cell that follows runs the four classical benchmarks under all four metrics on the validation window.


### 7.1 The four metrics

**Mean Absolute Error (MAE).**

$$\text{MAE} = \frac{1}{n}\sum_{t=1}^{n} \left| y_t - \hat{y}_t \right|$$

In the data's original units. Robust to outliers but treats large and small errors equally.

**Root Mean Squared Error (RMSE).**

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{t=1}^{n} \left( y_t - \hat{y}_t \right)^2}$$

Squaring before averaging penalizes large errors more than small ones. Sensitive to outliers; not in the data's original units (close-ish — same order of magnitude).

**Mean Absolute Percentage Error (MAPE).**

$$\text{MAPE} = \frac{100}{n}\sum_{t=1}^{n} \left| \frac{y_t - \hat{y}_t}{y_t} \right|$$

A scale-free percentage, comparable across series. Breaks if any $y_t \approx 0$ and is asymmetric — over-forecasts hurt more than under-forecasts of equal magnitude.

**Mean Absolute Scaled Error (MASE).**

$$\text{MASE} = \frac{\text{MAE}}{\text{MAE}_{\text{seasonal-naive, in-sample}}}$$

Scale-free; benchmarks against the in-sample seasonal-naive baseline. A value below 1 means the model beats the free baseline; above 1 means the seasonal-naive forecast is better than your model.


### 7.2 When to use which

| Use this metric ... | ... when |
|---|---|
| **MAE** | Reporting in business units (employees, dollars, units sold) and you want a robust default. |
| **RMSE** | Large misses cost much more than small ones (stock-outs, surge planning, safety-critical capacity). |
| **MAPE** | Reporting to non-technical audiences ("we are off by 3% on average") — and only when *y* is far from zero across the validation window. |
| **MASE** | Comparing forecasts across multiple series with different scales (cross-region demand, multi-product KPIs). |

**A question that often comes up here:** *"If the metrics rank models differently, which do I trust?"* The one whose error structure matches your business cost. If a stock-out costs 10× as much as overstock, RMSE is the honest metric — squaring penalizes the rare large miss exactly the way the cost matrix does. There is no "best" metric in the abstract; there is only the metric that aligns with consequences.


The four classical benchmarks are simple enough to implement by hand — that simplicity is the point. **Mean** forecasts the historical average for every future period (a flat line that ignores trend and seasonality entirely). **Naive** carries the last observed value forward unchanged, betting that tomorrow looks like today. **Seasonal-Naive** replays the most recent complete season (here, the last 12 months) forward, betting that next January looks like last January. **Drift** draws a straight line from the first training observation to the last and extends it forward — a simple trend extrapolation. The code cell below implements all four and evaluates each under the four metrics defined above.

In [ ]:
def forecast_mean(history, h):
    return np.full(h, history.mean())

def forecast_naive(history, h):
    return np.full(h, history.iloc[-1])

def forecast_seasonal_naive(history, h, season=12):
    last_season = history.iloc[-season:].values
    return np.tile(last_season, int(np.ceil(h / season)))[:h]

def forecast_drift(history, h):
    slope = (history.iloc[-1] - history.iloc[0]) / (len(history) - 1)
    return history.iloc[-1] + slope * np.arange(1, h + 1)

# Single-shot forecast on the validation window
horizon = len(df_val)
hist = df_train["y"]
preds = {
    "Mean":            forecast_mean(hist, horizon),
    "Naive":           forecast_naive(hist, horizon),
    "Seasonal-Naive":  forecast_seasonal_naive(hist, horizon),
    "Drift":           forecast_drift(hist, horizon),
}

# Plot all four against the validation actuals
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_train["ds"].iloc[-60:], df_train["y"].iloc[-60:], color="black", label="Train (last 60 mo)")
ax.plot(df_val["ds"], df_val["y"], color="black", linestyle="--", label="Val (actual)")
for name, p in preds.items():
    ax.plot(df_val["ds"], p, label=name)
ax.set_title("Four classical benchmarks on the validation window")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

# --- Multi-metric evaluation utility ---
def all_metrics(y_true, y_pred, training_y, season=12):
    """Return MAE, RMSE, MAPE, MASE for one (y_true, y_pred) pair."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mae = np.mean(np.abs(yt - yp))
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    mape = np.mean(np.abs((yt - yp) / yt)) * 100.0
    th = np.asarray(training_y, dtype=float)
    seasonal_naive_errors = np.abs(th[season:] - th[:-season])
    mae_naive = seasonal_naive_errors.mean()
    mase = mae / mae_naive if mae_naive > 0 else np.nan
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "MASE": mase}

# Evaluate each benchmark under all four metrics on the validation window
benchmark_table = pd.DataFrame({
    name: all_metrics(df_val["y"].values, p, df_train["y"].values)
    for name, p in preds.items()
}).T
print("Four classical benchmarks — single-shot validation, all four metrics:")
print(benchmark_table.round(3))

print("\\nRanking under each metric (1 = best):")
print(benchmark_table.rank(axis=0).astype(int))


**Reading the output:**

The plot tells you which baseline tracks the validation actuals best at a glance. The metric table and the ranking table together tell a richer story: the ranking can **agree across all four metrics** (the champion is robust — pick it with confidence) or it can **disagree** (one metric ranks Drift first while another ranks Seasonal-Naive first — that disagreement is the teaching moment).

When rankings disagree, the disagreement is almost always between **MAE** (treats every error equally) and **RMSE** (punishes large misses harder). A model that has small steady errors will win on MAE; a model that occasionally hits the trend perfectly but misses the seasonal swings badly will lose on RMSE. The choice between them is a **business** choice: does an employer’s procurement plan tolerate steady small misses or rare large ones?

**MASE** is the most cross-comparable: a value below 1 means the model beats the in-sample seasonal-naive baseline; a value above 1 means it does not. If your champion’s MASE is above 1, you cannot beat the free baseline — you do not have a champion.

These rankings depend on the data’s structural features. The next subsection makes that concrete by running the same benchmarks on a series with no seasonality at all.

---

### 7.3 Non-Seasonal Contrast — Google Daily Stock Prices

The US Retail Trade series above has clear annual seasonality, which is why **Seasonal-Naive** was such a strong baseline. But many business series — daily stock prices, intraday traffic, hourly server load, web-conversion rates — have little or no calendar seasonality. The classical benchmarks behave very differently there: **Drift** (linear extrapolation from start to end of training) often beats Seasonal-Naive by a lot because there is no annual pattern to lean on, and **Naive** (carry the last training value forward) can also be competitive because consecutive days are highly correlated.

To make the contrast concrete, we run the same benchmarks on Google daily closing prices: train on 2015, test on January 2016. The pedagogical point is that **the choice of benchmark depends on the data’s structural features, not on the model**.

In [ ]:
# Load Google daily closing prices from the GAFA stock dataset (same
# long-format columns as the employment data: unique_id, ds, y).
GAFA_URL = (
    "https://raw.githubusercontent.com/davi-moreira/"
    "2026Summer_predictive_analytics_purdue_MGMT474/main/"
    "lecture_slides/08_time_series/data/gafa_stock.csv"
)
gafa = pd.read_csv(GAFA_URL, parse_dates=["ds"])
goog = (
    gafa[gafa["unique_id"] == "GOOG_Close"]
    .loc[:, ["ds", "y"]]
    .sort_values("ds")
    .reset_index(drop=True)
)

# Train: 2015 (full calendar year). Test: January 2016.
goog_train = goog[(goog["ds"] >= "2015-01-01") & (goog["ds"] <  "2016-01-01")]
goog_test  = goog[(goog["ds"] >= "2016-01-01") & (goog["ds"] <  "2016-02-01")]
print(f"Train: {goog_train['ds'].min().date()} -> {goog_train['ds'].max().date()}  (n={len(goog_train)})")
print(f"Test : {goog_test['ds'].min().date()} -> {goog_test['ds'].max().date()}  (n={len(goog_test)})")

# Three classical benchmarks. Seasonal-Naive omitted because daily stock prices
# have no calendar seasonality (no 12-period annual cycle to anchor against).
horizon_g = len(goog_test)
hist_g = goog_train["y"]
preds_g = {
    "Mean":  forecast_mean(hist_g, horizon_g),
    "Naive": forecast_naive(hist_g, horizon_g),
    "Drift": forecast_drift(hist_g, horizon_g),
}

# Plot: 2015 training in grey + Jan 2016 test in black + three forecasts as
# colored lines extending past the training cutoff.
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(goog_train["ds"], goog_train["y"],
        color="grey", alpha=0.6, linewidth=0.8, label="2015 (train)")
ax.plot(goog_test["ds"],  goog_test["y"],
        color="black", linewidth=1.5, label="Jan 2016 (test, actual)")
for name, p in preds_g.items():
    ax.plot(goog_test["ds"], p, label=name, linewidth=1.2)
ax.axvline(pd.Timestamp("2016-01-01"), color="red", linestyle=":", alpha=0.5,
           label="Train / test boundary")
ax.set_title("Google Daily Closing Price — 2015 Train, Jan 2016 Test")
ax.set_xlabel("Date")
ax.set_ylabel("Closing price (USD)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

# Accuracy table — use season=1 (next-step random-walk baseline) for MASE
# because the series has no annual seasonality.
goog_table = pd.DataFrame({
    name: all_metrics(goog_test["y"].values, p, goog_train["y"].values, season=1)
    for name, p in preds_g.items()
}).T
print("\nAccuracy table — Google daily, January 2016 horizon (MASE referenced to 1-step naive):")
print(goog_table.round(3))
print("\nRanking under each metric (1 = best):")
print(goog_table.rank(axis=0).astype(int))

**Reading the output:**

The ranking on Google daily prices typically **flips** relative to US Retail Trade. Seasonal-Naive does not appear because there is no annual seasonality to anchor against. Among the three remaining benchmarks, **Drift** usually wins because daily prices follow an approximate random walk with mild long-run drift — the slope from year-start to year-end is a reasonable extrapolation for the first few weeks of the next year. **Naive** is competitive because consecutive trading days are highly correlated; tomorrow’s price is most likely close to today’s. **Mean** is usually worst — averaging a year of prices and projecting the mean forward throws away the level you actually closed at.

The business takeaway is that **the right benchmark depends on the data, not the model.** On retail employment, Seasonal-Naive was the strong baseline; on daily stock, Drift is. Build the classical benchmarks first; let the data tell you which one a learned model has to beat. Note also that **MASE with season = 1** is the natural denominator for non-seasonal series — same formula, different reference baseline (1-step random walk instead of 12-step seasonal anchor). A MASE below 1 still means “the model beats the free baseline”; the baseline is just a different one.

With the benchmarks established on both seasonal and non-seasonal series, section 8 asks: can a simple learned model — linear regression on lag features — beat those free baselines?

---

## 8. Lag Features + Linear Regression

The cheapest, most useful features for any business time series are **lags**: last month's value (`lag1`) and 12 months ago (`lag12`). A linear regression on `[lag1, lag12]` is a strong, interpretable baseline that captures both short-term momentum and annual seasonality without any deep-learning machinery.


In [ ]:
def add_lags(frame, lags=(1, 12)):
    out = frame.copy().sort_values("ds").reset_index(drop=True)
    for L in lags:
        out[f"lag{L}"] = out["y"].shift(L)
    return out

# Build lag features on the FULL series so train/val/test rows can pull lags from earlier rows
df_lag = add_lags(df).dropna()

# Re-split on the lagged frame using the same date boundaries
train_max_date = df_train["ds"].max()
val_max_date   = df_val["ds"].max()
df_lag_train = df_lag[df_lag["ds"] <= train_max_date].copy()
df_lag_val   = df_lag[(df_lag["ds"] > train_max_date) & (df_lag["ds"] <= val_max_date)].copy()
df_lag_test  = df_lag[df_lag["ds"] > val_max_date].copy()

print(f"Lagged train: n={len(df_lag_train)}  (lost {len(df_train) - len(df_lag_train)} rows to lag12)")
print(f"Lagged val  : n={len(df_lag_val)}")
print(f"Lagged test : n={len(df_lag_test)}")

X_train_lag = df_lag_train[["lag1", "lag12"]].values
y_train_lag = df_lag_train["y"].values
X_val_lag   = df_lag_val[["lag1", "lag12"]].values
y_val_lag   = df_lag_val["y"].values

lr = LinearRegression().fit(X_train_lag, y_train_lag)
y_pred_train = lr.predict(X_train_lag)
y_pred_val   = lr.predict(X_val_lag)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(df_lag_train["ds"], y_train_lag, color="#1f77b4", label="Actual", linewidth=0.8)
axes[0].plot(df_lag_train["ds"], y_pred_train, color="#2ca02c", label="Linear [lag1, lag12]", linewidth=0.8)
axes[0].set_title("Training fit")
axes[0].legend()
axes[1].plot(df_lag_val["ds"], y_val_lag, color="black", label="Actual")
axes[1].plot(df_lag_val["ds"], y_pred_val, color="#2ca02c", label="Linear [lag1, lag12]")
axes[1].set_title("Validation fit")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"\nLinear coefficients : lag1={lr.coef_[0]:.3f}, lag12={lr.coef_[1]:.3f}")
print(f"Linear intercept   : {lr.intercept_:.1f}")
print(f"Validation MAE     : {mean_absolute_error(y_val_lag, y_pred_val):.1f}")


**Reading the output:**

The training fit (left) tracks the actuals tightly — that is the visual confirmation that lag1 + lag12 carry most of the forecastable signal. The validation fit (right) is the honest test; it should also track closely if the model generalizes.

The coefficients tell the workforce planner a clean story: each unit of last month’s employment contributes its coefficient × itself to next month’s prediction; each unit of employment 12 months ago contributes its coefficient × itself. Add the intercept and you have a forecast any analyst can verify by hand.

> **A question that often comes up here:** *“Why don’t the coefficients add to 1?”* They roughly do, but not exactly — the exact sum depends on how much of next month’s variance is explained by short-term momentum (`lag1`) vs. the seasonal anchor (`lag12`). When the trend is strong (as here), `lag1` dominates; in highly seasonal series with little trend, `lag12` dominates.

> **A question that often comes up at this point:** *“Shouldn’t I difference the series first to make it stationary?”* That concern comes from the ARIMA tradition, where the model assumes a stationary process and differencing is the mechanism that achieves it. Here we take a different approach: a regression model with lag features in levels. The model learns the relationship between this month’s value and last month’s (and last year’s) directly — no explicit differencing required. Both frameworks can work; the lag-feature regression is more transparent for a first encounter because the coefficients have a plain-language interpretation (“each unit of last month’s employment contributes X to next month’s forecast”).

The single-split validation MAE gives one number, but that is one roll of the dice. Section 9 runs all five candidates — the four classical benchmarks plus the linear model — on the same walk-forward folds and applies the Student’s *t* 95% CI decision rule from nb08.

---

## 9. Cross-Validated Comparison — Five Candidates, Identical Folds

We now compare five forecasters on the **same** walk-forward folds, so the comparison is honest. Each fold's MAE is one observation; the mean and Student's *t* 95% CI quantify how confident we are in the ranking.


In [ ]:
def cv_score_func(history_to_pred, df_lag_window, splits, score_fn=None):
    """Run a forecaster across walk-forward folds and return per-fold MAEs."""
    if score_fn is None:
        score_fn = mean_absolute_error
    scores = []
    for tr, va in splits.split(df_lag_window):
        train_chunk = df_lag_window.iloc[tr]
        val_chunk = df_lag_window.iloc[va]
        y_pred = history_to_pred(train_chunk, val_chunk)
        scores.append(score_fn(val_chunk["y"], y_pred))
    return np.array(scores)

def pred_naive(train, val):           return val["lag1"].values
def pred_seasonal_naive(train, val):  return val["lag12"].values
def pred_mean(train, val):            return np.full(len(val), train["y"].mean())
def pred_linear(train, val):
    m = LinearRegression().fit(train[["lag1", "lag12"]], train["y"])
    return m.predict(val[["lag1", "lag12"]])
def pred_ridge(train, val):
    m = Ridge(alpha=1.0, random_state=RANDOM_SEED).fit(train[["lag1", "lag12"]], train["y"])
    return m.predict(val[["lag1", "lag12"]])

splits = TimeSeriesSplit(n_splits=5)
candidates = {
    "Mean":             pred_mean,
    "Naive (lag1)":     pred_naive,
    "Seasonal-Naive (lag12)": pred_seasonal_naive,
    "Linear [lag1, lag12]":   pred_linear,
    "Ridge  [lag1, lag12]":   pred_ridge,
}

# --- Selection metric (MAE) with Student's t 95% CI ---
results = pd.DataFrame({name: cv_score_func(fn, df_lag_train, splits) for name, fn in candidates.items()})
t_crit = student_t.ppf(0.975, df=4)
summary = pd.DataFrame({
    "MAE_mean": results.mean(),
    "MAE_sd":   results.std(ddof=1),
    "CI_halfwidth": results.std(ddof=1) / np.sqrt(5) * t_crit,
}).sort_values("MAE_mean")
summary["CI_low"] = summary["MAE_mean"] - summary["CI_halfwidth"]
summary["CI_high"] = summary["MAE_mean"] + summary["CI_halfwidth"]
print("Selection metric (MAE) — 5-fold walk-forward CV with 95% CI:")
print(summary.round(2))

# --- Multi-metric sensitivity check (do other metrics agree on the ranking?) ---
def per_fold_all_metrics(history_to_pred, df_lag_window, splits, season=12):
    out = {"MAE": [], "RMSE": [], "MAPE": [], "MASE": []}
    for tr, va in splits.split(df_lag_window):
        train_chunk = df_lag_window.iloc[tr]
        val_chunk = df_lag_window.iloc[va]
        yt = val_chunk["y"].values
        yp = history_to_pred(train_chunk, val_chunk)
        out["MAE"].append(np.mean(np.abs(yt - yp)))
        out["RMSE"].append(np.sqrt(np.mean((yt - yp) ** 2)))
        out["MAPE"].append(np.mean(np.abs((yt - yp) / yt)) * 100.0)
        th = train_chunk["y"].values
        if len(th) > season:
            mae_naive = np.mean(np.abs(th[season:] - th[:-season]))
            out["MASE"].append(np.mean(np.abs(yt - yp)) / mae_naive)
        else:
            out["MASE"].append(np.nan)
    return {k: np.array(v) for k, v in out.items()}

all_results = {name: per_fold_all_metrics(fn, df_lag_train, splits) for name, fn in candidates.items()}
metric_means = pd.DataFrame({m: {name: r[m].mean() for name, r in all_results.items()}
                             for m in ["MAE", "RMSE", "MAPE", "MASE"]})
metric_means = metric_means.loc[summary.index]  # match selection ordering
print("\\nMulti-metric sensitivity check — per-candidate mean across all four metrics:")
print(metric_means.round(3))
print("\\nRanking under each metric (1 = best):")
print(metric_means.rank(axis=0).astype(int))

# --- Selection bar chart (MAE with 95% CI) ---
fig, ax = plt.subplots(figsize=(11, 5))
y_pos = np.arange(len(summary))
ax.barh(y_pos, summary["MAE_mean"],
        xerr=summary["CI_halfwidth"], color="#1f77b4", edgecolor="black", capsize=4)
ax.set_yticks(y_pos)
ax.set_yticklabels(summary.index)
ax.invert_yaxis()
ax.set_xlabel("MAE (5-fold walk-forward CV; bars = 95% CI)")
ax.set_title("Five-candidate forecast comparison — selection metric (MAE)")
plt.tight_layout()
plt.show()


**Reading the output:**

Two tables give two views of the same comparison. The **MAE selection table** with 95% CI half-widths is the **primary decision rule** — lowest mean MAE wins, and non-overlapping CIs against the runner-up means the win is statistically real. Use this table to pick the champion that goes into the locked-test ceremony. The **multi-metric sensitivity table** asks *“would I pick a different champion if I cared about RMSE / MAPE / MASE instead?”* If the ranking matrix shows the same model ranked #1 across all four metrics, the champion is **robust** — ship it. If the rankings disagree (model A is #1 on MAE but model B is #1 on RMSE), reach for the metric whose error structure matches the business cost.

Three interpretation rules borrowed from nb08:

1. **Non-overlapping CIs** between candidate A and candidate B → A is genuinely better on the selection metric.
2. **Overlapping CIs** → no statistical evidence to prefer one over the other; pick the simpler model (Occam’s razor).
3. **Mean is far worse than the rest** → expected. It ignores trend and seasonality entirely; it is only here as a sanity floor.

If the linear and Ridge models have overlapping CIs, **Ridge does not earn its place** here — the regularization adds machinery without a measurable payoff. That is the right outcome for a 2-feature model; Ridge typically wins when the feature count is large and multicollinearity is a real risk.

---

## 📝 PAUSE-AND-DO Exercise 1 — Add `lag2` and `lag6` (10 minutes)

**Task:** Engineer two more lag features (`lag2`, `lag6`) and rerun the comparison. Does the four-feature linear regression beat the two-feature baseline by **non-overlapping CIs**?

**Hints:**
- Use `add_lags(df, lags=(1, 2, 6, 12))` to extend the lag list.
- Recompute `df_lag_train` from the new lagged frame (same date boundaries as before).
- Add a sixth row to the comparison table — call it `Linear [lag1, lag2, lag6, lag12]`.
- Compare its CI to the original `Linear [lag1, lag12]`. Overlap = the new features did not earn their place.

Type your code in the cell below.


> 💡 **Gemini Prompt:** *"I have a walk-forward CV comparison of five forecasters (Mean, Naive, Seasonal-Naive, Linear [lag1, lag12], Ridge [lag1, lag12]) on monthly US retail employment data, using TimeSeriesSplit(n_splits=5). The helper function add_lags(df, lags) creates lagged columns, and cv_score_func(pred_fn, df_lag_train, splits) returns per-fold MAEs. I want to add two more lag features — lag2 and lag6 — and see whether a four-feature LinearRegression beats the two-feature version. Use add_lags(df, lags=(1, 2, 6, 12)).dropna() to build the extended feature set, re-split using train_max_date, define a pred_linear_v2 function using features ['lag1', 'lag2', 'lag6', 'lag12'], run cv_score_func on the same TimeSeriesSplit folds, and build an updated MAE summary table with Student's t 95% CIs (t_crit is already defined). Print the table and a horizontal bar chart comparing all six candidates."*
>
> **After running, verify:**
> - [ ] The new model `Linear [lag1, lag2, lag6, lag12]` appears in the summary table alongside the original five candidates
> - [ ] The CI for the four-feature model overlaps (or does not overlap) with `Linear [lag1, lag12]` — note which
> - [ ] The bar chart shows error bars for all six candidates
> - [ ] No test-set data was used anywhere

In [ ]:
# YOUR SOLUTION CODE HERE

# Hints:
# df_lag_v2 = add_lags(df, lags=(1, 2, 6, 12)).dropna()
# df_lag_v2_train = df_lag_v2[df_lag_v2["ds"] <= train_max_date]
# def pred_linear_v2(train, val):
#     features = ["lag1", "lag2", "lag6", "lag12"]
#     m = LinearRegression().fit(train[features], train["y"])
#     return m.predict(val[features])
# results["Linear [lag1, lag2, lag6, lag12]"] = cv_score_func(pred_linear_v2, df_lag_v2_train, splits)
# Build the new summary, compare CIs.


## 10. Opening the Locked Test Window — One-Shot Evaluation

We now do the time-series analog of nb14’s “test-set opening ceremony.” The ritual is the same: pick the champion, refit on all of train + val, predict the locked window once, and read the verdict — **INSIDE / ABOVE / BELOW** the CV 95% CI. What is new here is the **prediction interval**: we estimate the residual sigma from walk-forward folds (not from the final training fit) and wrap a 95% Gaussian band around each point forecast. The workforce planner gets not just “forecast = X” but “forecast = X ± Y with Z% empirical coverage” — a deliverable the legislature can read.

In [ ]:
# Step 1: estimate residual sigma from walk-forward training fits.
# Each fold's residuals come from a fit that did NOT see the validation rows.
fold_residuals = []
for tr, va in splits.split(df_lag_train):
    train_chunk = df_lag_train.iloc[tr]
    val_chunk   = df_lag_train.iloc[va]
    m_fold = LinearRegression().fit(train_chunk[["lag1", "lag12"]], train_chunk["y"])
    pred_fold = m_fold.predict(val_chunk[["lag1", "lag12"]])
    fold_residuals.extend(val_chunk["y"].values - pred_fold)
fold_residuals = np.array(fold_residuals)
sigma_residual = fold_residuals.std(ddof=1)
print(f"Walk-forward residual sigma: {sigma_residual:.2f} (units: thousands of employees)")

# Step 2: refit champion on train + val (lag features only — no test-set leak)
df_lag_trainval = pd.concat([df_lag_train, df_lag_val])
champion = LinearRegression().fit(
    df_lag_trainval[["lag1", "lag12"]], df_lag_trainval["y"]
)

# Step 3: point forecast on the locked test window
y_test_pred = champion.predict(df_lag_test[["lag1", "lag12"]])
test_mae = mean_absolute_error(df_lag_test["y"], y_test_pred)

# Step 4: 95% prediction interval (Gaussian assumption on residuals)
z_95 = 1.96
y_test_lower = y_test_pred - z_95 * sigma_residual
y_test_upper = y_test_pred + z_95 * sigma_residual

# Step 5: empirical coverage of the 95% PI on the locked test window
inside = ((df_lag_test["y"].values >= y_test_lower) &
          (df_lag_test["y"].values <= y_test_upper)).mean()

# Pull the champion's CV CI for the verdict
champ_row = summary.loc["Linear [lag1, lag12]"]
cv_low, cv_high = champ_row["CI_low"], champ_row["CI_high"]
verdict = ("INSIDE the CV 95% CI" if cv_low <= test_mae <= cv_high
           else "ABOVE the CV 95% CI (overfitting?)" if test_mae > cv_high
           else "BELOW the CV 95% CI (lucky test window?)")

print(f"\\nChampion: Linear [lag1, lag12]")
print(f"CV MAE 95% CI : [{cv_low:.2f}, {cv_high:.2f}]")
print(f"Test MAE      : {test_mae:.2f}  ->  {verdict}")
print(f"Empirical 95% PI coverage on test: {inside*100:.1f}%  (nominal: 95.0%)")

# Plot: train + val + test_actual + champion_forecast + shaded 95% PI band
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df_lag_train["ds"], df_lag_train["y"], color="#1f77b4", label="Train", linewidth=0.8)
ax.plot(df_lag_val["ds"],   df_lag_val["y"],   color="#ff7f0e", label="Val", linewidth=0.8)
ax.plot(df_lag_test["ds"],  df_lag_test["y"],  color="#d62728", label="Test (actual)", linewidth=1.5)
ax.plot(df_lag_test["ds"],  y_test_pred,       color="#2ca02c", linestyle="--",
        label="Champion forecast", linewidth=1.5)
ax.fill_between(df_lag_test["ds"], y_test_lower, y_test_upper,
                color="#2ca02c", alpha=0.20, label="95% prediction interval")
ax.set_title("Locked Test Window — Forecast + 95% Prediction Interval")
ax.legend()
plt.tight_layout()
plt.show()


**Reading the output:**

Two diagnostics, one verdict.

**The point-forecast verdict** mirrors nb14's protocol: **INSIDE** the CV CI means the CV-based selection generalized; **ABOVE** signals overfitting on the training history; **BELOW** is unusual but possible (the locked window happened to be easier than the average CV fold).

**The prediction-interval coverage** is the new diagnostic. We constructed the 95% PI under a **Gaussian assumption** on the walk-forward residuals: $\hat{y}_t \pm 1.96 \cdot \sigma_{\text{residual}}$. Then we asked *"what fraction of locked-test actuals actually fell inside that interval?"*. Three reading rules:

- **Coverage near 95%** (say, 90–98%): the Gaussian assumption holds, the interval is honest, the workforce planner can quote it.
- **Coverage well below 95%** (say, 70–85%): the model is **overconfident** — the residuals have heavier tails than Gaussian, or there are structural breaks the lag-feature model cannot capture (recessions, policy shocks). Widen the interval before reporting.
- **Coverage near 100%**: the interval is **too wide** — typically because the residual sigma was inflated by a few outlier folds. Tighter intervals would still cover the right amount and be more useful.

For a workforce planner, the deliverable is no longer just *"forecast = X"* but *"forecast = X, 95% interval [X−Y, X+Y], and the model has been cross-validated to deliver that coverage on held-out data."* That is the line that goes on the M4 poster.

**A question that often comes up here:** *"Why use the residual sigma from CV folds instead of from the final training fit?"* Because the final training residuals are in-sample — the model fitted those rows. Walk-forward residuals are out-of-sample; they reflect the noise the model will encounter on truly future data. Using in-sample residuals would systematically underestimate sigma and produce overconfident intervals. Same principle as nb08's CV CIs.

---


## 📝 PAUSE-AND-DO Exercise 2 — Add Ridge Tuning (10 minutes)

**Task:** Ridge with `alpha=1.0` may be over- or under-regularized. Sweep `alpha ∈ [0.01, 0.1, 1, 10, 100]`, run walk-forward CV at each alpha, and pick the alpha that minimizes mean MAE. Compare its CI to the unregularized linear baseline.

**Hints:**
- Loop over alphas; each iteration runs `cv_score_func(...)` with a new `pred_ridge_alpha`.
- Build a small `pd.DataFrame` of `alpha` vs. `MAE_mean` and `CI_halfwidth`.
- The plot to make: alpha on a log x-axis, MAE on the y-axis, with error bars for the CI.

Type your code in the cell below.

**Bonus — interpret the prediction-interval coverage:** look at the empirical coverage printed in section 10 (`X.X%`). Is it close to the nominal 95%? If not, write one sentence explaining what that says about the residual distribution (heavy tails? structural break? Gaussian approximation failing?).


> 💡 **Gemini Prompt:** *"I have a walk-forward CV setup using TimeSeriesSplit(n_splits=5) on monthly US retail employment data with lag1 and lag12 features in df_lag_train. The helper cv_score_func(pred_fn, df_lag_train, splits) returns per-fold MAEs, and t_crit is already defined. Sweep Ridge alpha over [0.01, 0.1, 1, 10, 100] — at each alpha, define a prediction function that fits Ridge(alpha=alpha, random_state=RANDOM_SEED) on ['lag1', 'lag12'], run cv_score_func, and collect the mean MAE and CI half-width. Build a DataFrame of alpha vs MAE_mean and CI_halfwidth. Print the table, plot alpha on a log x-axis vs MAE with error bars using ax.errorbar, and compare the best Ridge CI to the unregularized LinearRegression CI from the summary table to determine whether Ridge earns its place."*
>
> **After running, verify:**
> - [ ] The table shows five rows (one per alpha) with MAE_mean and CI half-width columns
> - [ ] The plot has alpha on a log-scaled x-axis with error bars at each point
> - [ ] A printed comparison states whether the best Ridge CI overlaps with the Linear CI
> - [ ] All evaluation uses walk-forward CV on training data only — no test-set leak

In [ ]:
# YOUR SOLUTION CODE HERE

# Hints:
# alphas = [0.01, 0.1, 1, 10, 100]
# rows = []
# for a in alphas:
#     def pred_ridge_a(train, val, alpha=a):
#         m = Ridge(alpha=alpha, random_state=RANDOM_SEED).fit(train[["lag1","lag12"]], train["y"])
#         return m.predict(val[["lag1","lag12"]])
#     fold_maes = cv_score_func(pred_ridge_a, df_lag_train, splits)
#     rows.append({"alpha": a, "MAE_mean": fold_maes.mean(), "CI_hw": fold_maes.std(ddof=1)/np.sqrt(5)*t_crit})
# Then plot with errorbar() on a log-x axis.


## 11. Forecast Accuracy Diagnostics — Residuals and Horizon

Two questions are worth answering before we wrap up. First: **why did we estimate the prediction-interval sigma from walk-forward residuals instead of in-sample training residuals?** A side-by-side comparison answers it visually. Second: **does forecast error grow as we predict further ahead?** A rolling-forecast-origin sweep across horizons answers that one directly.

These two diagnostics complete the toolkit a workforce planner needs to defend a forecast: the **point forecast** (§7-9), the **prediction interval** (§10), the **residual diagnostic** (§11.1), and the **horizon curve** (§11.2).


### 11.1 In-sample residuals vs walk-forward residuals

If the champion is fit on training data and we read its residuals on those same rows, we get **in-sample** residuals. Those residuals are systematically smaller than residuals on rows the model has not seen. The in-sample / out-of-sample gap is exactly the gap a prediction interval has to honor — and it is the reason §10's PI used walk-forward residual sigma rather than the easier-to-compute in-sample sigma.


In [ ]:
# Champion fit on the full lagged training data (the "in-sample" world)
champ_in_sample = LinearRegression().fit(
    df_lag_train[["lag1", "lag12"]], df_lag_train["y"]
)
in_sample_pred = champ_in_sample.predict(df_lag_train[["lag1", "lag12"]])
in_sample_residuals = df_lag_train["y"].values - in_sample_pred
in_sample_mae = float(np.mean(np.abs(in_sample_residuals)))
in_sample_sigma = float(in_sample_residuals.std(ddof=1))

# Walk-forward residuals already computed in §10 as `fold_residuals`,
# `sigma_residual` — reuse them.
cv_mae = float(np.mean(np.abs(fold_residuals)))
cv_sigma = float(sigma_residual)

cmp_table = pd.DataFrame({
    "MAE":             [in_sample_mae, cv_mae],
    "Residual sigma":  [in_sample_sigma, cv_sigma],
}, index=["In-sample (training fit)", "Walk-forward CV (out-of-sample)"])
print("Residual diagnostics — same champion, two ways of measuring its noise:")
print(cmp_table.round(2))
print(f"\nRatio (CV / in-sample) MAE   : {cv_mae / in_sample_mae:.2f}x")
print(f"Ratio (CV / in-sample) sigma : {cv_sigma / in_sample_sigma:.2f}x")

# Two-panel plot: residual histograms overlaid + MAE bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(in_sample_residuals, bins=40, color="#1f77b4", alpha=0.6, label="In-sample")
axes[0].hist(fold_residuals,      bins=40, color="#d62728", alpha=0.6, label="Walk-forward CV")
axes[0].axvline(0, color="black", linewidth=0.5)
axes[0].set_xlabel("Residual (units: thousands of employees)")
axes[0].set_ylabel("Count")
axes[0].set_title("Residual distributions")
axes[0].legend()

axes[1].bar(["In-sample\\n(training fit)", "Walk-forward CV\\n(out-of-sample)"],
            [in_sample_mae, cv_mae],
            color=["#1f77b4", "#d62728"], edgecolor="black")
axes[1].set_ylabel("Mean absolute residual")
axes[1].set_title("Residual MAE — why the PI used CV, not in-sample")
plt.tight_layout()
plt.show()


**Reading the output:**

The walk-forward residuals are wider and more dispersed than the in-sample residuals — the CV histogram has a heavier spread, and the CV MAE is typically **1.5× to 3× larger** than the in-sample MAE on a series with strong autocorrelation like this one. That gap is the whole reason §10's prediction interval used the walk-forward sigma: a PI built from in-sample sigma would have been **systematically too narrow**, claiming 95% coverage on paper while letting more than 5% of true future values fall outside the band.

This is the time-series version of the lesson nb08 taught for k-fold: **in-sample evaluation is overconfident; out-of-sample evaluation is honest**. The same principle applies to point estimates (MAE) and to uncertainty estimates (sigma).


### 11.2 Forecast horizon and accuracy

A 1-month-ahead forecast is much easier than a 12-month-ahead forecast. We sweep horizon $h \in \{1, 2, \dots, 12\}$ using a **rolling forecast origin** (each `TimeSeriesSplit` fold supplies one cutoff) combined with **recursive multi-step forecasting** (the model's own predictions feed back as lag inputs for further-out steps).


In [ ]:
def recursive_forecast(model, history_y, h, season=12):
    """Forecast h steps ahead recursively. `history_y` must contain at least
    the last `season` observed values; predictions feed back as lag1 inputs."""
    history = list(history_y)
    preds = []
    for _ in range(h):
        lag1  = history[-1]
        lag12 = history[-season]
        x = np.array([[lag1, lag12]])
        yhat = float(model.predict(x)[0])
        preds.append(yhat)
        history.append(yhat)
    return np.array(preds)

# For each TS-CV fold, fit on training, recursive-forecast h=1..12,
# collect squared errors per horizon.
HORIZONS = list(range(1, 13))
errors_by_h = {h: [] for h in HORIZONS}

for tr, va in TimeSeriesSplit(n_splits=5).split(df_lag_train):
    train_chunk = df_lag_train.iloc[tr]
    val_chunk   = df_lag_train.iloc[va]
    m = LinearRegression().fit(train_chunk[["lag1", "lag12"]], train_chunk["y"])
    history = train_chunk["y"].values  # actual history up to the cutoff
    h_max = min(len(HORIZONS), len(val_chunk))
    preds = recursive_forecast(m, history, h_max)
    actuals = val_chunk["y"].values[:h_max]
    for i, (a, p) in enumerate(zip(actuals, preds), start=1):
        if i in errors_by_h:
            errors_by_h[i].append((a - p) ** 2)

rmse_by_h = {h: float(np.sqrt(np.mean(es))) for h, es in errors_by_h.items() if es}

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(rmse_by_h.keys()), list(rmse_by_h.values()),
        "o-", color="#9467bd", linewidth=2, markersize=8)
ax.set_xlabel("Forecast horizon h (months ahead)")
ax.set_ylabel("RMSE (recursive forecast)")
ax.set_title("Forecast accuracy degrades with horizon — Linear [lag1, lag12]")
ax.grid(alpha=0.3)
ax.set_xticks(list(rmse_by_h.keys()))
plt.tight_layout()
plt.show()

rmse_table = pd.Series(rmse_by_h, name="RMSE").round(2).to_frame()
rmse_table.index.name = "h (months ahead)"
print(rmse_table)


**Reading the output:**

The line typically rises monotonically: 1-month-ahead RMSE is the smallest; 12-month-ahead RMSE is multiple times larger. Two effects compound. **Recursive feedback** is the first: for horizons beyond one month, the model uses its own imperfect predictions as `lag1` inputs, so errors at step 1 propagate into step 2, step 2 into step 3, and so on — the forecast walks further from the truth at each step. **Information staleness** is the second: the model has access to actual lag values up to the cutoff but no information about events after the cutoff, and the further out it forecasts, the more such events accumulate — recessions, policy shocks, structural changes.

Practical implication for the workforce planner: the forecast they quote **one month out** can carry tight confidence; the forecast they quote **one year out** cannot. The horizon-vs-RMSE curve is the right artifact for that conversation — and a strong candidate figure for the M4 poster’s “Limitations” section.

> **A question that often comes up here:** *“Why does this differ from the §9 CV table?”* The §9 table averages across all rows in each validation fold, so it implicitly averages over many horizons mixed together. §11.2 separates them — one RMSE per horizon — which is what you actually need when the business question is *“how far ahead can we trust this forecast?”*

With the point forecast, the prediction interval, the residual diagnostic, and the horizon curve all in hand, you have the full toolkit the workforce planner needs to defend a forecast. Section 12 pulls it all together.

---

## 12. Wrap-Up — Key Takeaways

1. **Forecasting is supervised learning with one structural rule: never let the future leak into the past.** That single rule changes the train/test split (recent slice held out), the cross-validation strategy (`TimeSeriesSplit`), and what counts as a feature (lags, not random shuffling).
2. **The Week-1 analytics workflow ports cleanly to time series.** EDA → split → baselines → linear features → regularization is the same recipe; only the partition strategy and feature engineering change.
3. **Naive baselines are surprisingly hard to beat.** If your fancy model does not beat seasonal-naive on identical CV folds with non-overlapping CIs, you do not have a champion — you have noise.
4. **The cost of lag features is the loss of the earliest rows.** A 12-month seasonal lag costs you the first year of history. Plan for it.
5. **Walk-forward CV is the time-series spine of CV-first evaluation,** exactly like `StratifiedKFold` was the classification spine in nb08–nb14.

**A question that often comes up here:** *"Where do RNNs and transformers fit?"* They are alternatives to lag-feature linear models when (a) the series is long enough (thousands of points, not 960), (b) the dependence is highly non-linear, and (c) you can spare an order of magnitude more compute. For business problems with a few decades of monthly history, a well-engineered lag-feature linear regression is almost always the right starting point — and often the right ending point. Deep learning gets the awareness module it deserves in **nb19**.

**Next stop — nb17: Data Communication and Poster Design.** Now that you have a forecast, a defensible CV-based comparison, and a clean test-set ceremony verdict, the question becomes how to **communicate** them: the six principles of data communication, the eleven-section poster architecture for the M4 deliverable, and the data-ink-ratio cleanup that turns a notebook plot into a poster figure.

---


## Participation Assignment Submission Instructions

1. **Complete both PAUSE-AND-DO exercises** (sections after 9 and 10).
2. **Run all cells** (`Runtime → Run all`).
3. **Save with output** (`File → Download → Download .ipynb`).
4. **Submit to Brightspace** as `nb16_time_series_forecasting_<your_lastname>.ipynb`.

**Bibliography**
- Hyndman & Athanasopoulos: *Forecasting: Principles and Practice* (FPP3) — the [free online textbook](https://otexts.com/fpp3/) is the deep dive on every concept above.
- scikit-learn User Guide: `TimeSeriesSplit` and time-series cross-validation.
- statsmodels: `STL` decomposition and the autocorrelation function.

<center>

# Thank you!

</center>
